<a href="https://colab.research.google.com/github/Ranjith1816/Gen_AI--Assignment-_task/blob/main/Sentiment_Analysis_using_NLP_Pipeline_%26_ML_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Loading dataset

In [14]:
from google.colab import files
uploaded = files.upload()

Saving IMDB_Dataset.csv to IMDB_Dataset.csv


1. understanding data

In [16]:
import pandas as pd
df = pd.read_csv("IMDB_Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [18]:
print("Shape of dataset:", df.shape)

# printing column names
print("\nColumns:")
print(df.columns)

# checking class distribution
print("\nClass Distribution:")
print(df['sentiment'].value_counts())

Shape of dataset: (50000, 2)

Columns:
Index(['review', 'sentiment'], dtype='object')

Class Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


2. NLP preprocessing

In [20]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# download stopwords (only once)
nltk.download('stopwords')

# loading stopwords
stop_words = set(stopwords.words('english'))

# using stemming
stemmer = PorterStemmer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [22]:
def preprocess_text(text):

    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)

    # tokenization
    words = text.split()

    # remove stopwords
    words = [w for w in words if w not in stop_words]

    # stemming
    words = [stemmer.stem(w) for w in words]

    return " ".join(words)
df['processed_text'] = df['review'].apply(preprocess_text)
df[['review', 'processed_text']].head()

,review,processed_text
0,One of the other reviewers has mentioned that ...,one review mention watch oz episod youll hook ...
1,A wonderful little production. <br /><br />The...,wonder littl product br br film techniqu unass...
2,I thought this was a wonderful way to spend ti...,thought wonder way spend time hot summer weeke...
3,Basically there's a family where a little boy ...,basic there famili littl boy jake think there ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visual stun film...


3. Feature engineering

In [24]:
# input features (text)
X = df['processed_text']

# target labels
y = df['sentiment']
# converting text into numerical features using Bag of Words
from sklearn.feature_extraction.text import CountVectorizer

# creating BoW model
bow = CountVectorizer()

# converting text into vectors
X_bow = bow.fit_transform(X)

print("BoW Shape:", X_bow.shape)

BoW Shape: (50000, 137463)


In [25]:
#converting text using TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

# creating TF-IDF model
tfidf = TfidfVectorizer()
# transforming text
X_tfidf = tfidf.fit_transform(X)
print("TF-IDF Shape:", X_tfidf.shape)

TF-IDF Shape: (50000, 137463)


4. Model building

In [26]:
#logestic regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

lr = LogisticRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Calculate and print the accuracy score
accuracy = accuracy_score(y_test, y_pred_lr)
print(f"Accuracy of Logistic Regression model: {accuracy:.4f}")

Accuracy of Logistic Regression model: 0.8898


In [28]:
#naive bayes
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
print(y_pred_nb[:10])  # first 10 predictions


['positive' 'positive' 'negative' 'positive' 'negative' 'positive'
 'positive' 'negative' 'negative' 'negative']


In [30]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

5. Model evaluation

In [32]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(name, y_test, y_pred):
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    # Specify pos_label for string labels
    print("Precision:", precision_score(y_test, y_pred, pos_label='positive'))
    print("Recall:", recall_score(y_test, y_pred, pos_label='positive'))
    print("F1 Score:", f1_score(y_test, y_pred, pos_label='positive'))
evaluate("Logistic Regression", y_test, y_pred_lr)
evaluate("Naive Bayes", y_test, y_pred_nb)
evaluate("Decision Tree", y_test, y_pred_dt)


Logistic Regression
Accuracy: 0.8898
Precision: 0.8800926819849392
Recall: 0.9045445524905735
F1 Score: 0.8921511058915639

Naive Bayes
Accuracy: 0.8622
Precision: 0.8751793400286944
Recall: 0.8473903552292121
F1 Score: 0.8610606977213148

Decision Tree
Accuracy: 0.7239
Precision: 0.7300080775444265
Recall: 0.7174042468743799
F1 Score: 0.7236512861575418


6.comparision

Best Preprocessing Steps

The preprocessing pipeline included lowercasing, removal of punctuation and special characters, stopword removal, tokenization, and stemming.

>Lowercasing ensured uniformity (e.g., “Good” and “good” treated the same)

>Removing punctuation & special characters reduced noise in text

>Stopword removal eliminated common words like “the”, “is”, which do not contribute to sentiment

>Tokenization helped break text into meaningful units (words)
Stemming reduced words to root form (e.g., “liked”, “liking” → “like”)

Insight:

These steps significantly reduced dimensionality and noise, leading to better model performance.

###Best Vectorization Technique

TF-IDF performed better than Bag of Words (BoW).

Why TF-IDF is better:

Assigns importance to rare but meaningful words
Reduces weight of common words
Handles high-dimensional sparse data efficiently

Example Insight:

Words like “excellent”, “worst” get higher importance than common words like “movie”.

Conclusion:

TF-IDF improved classification accuracy compared to BoW.

###Best Model

Logistic Regression achieved the best performance among all models.

Why?

Works well with high-dimensional sparse features
Handles linear separability effectively
More stable compared to tree-based models in text data

Insight:

Logistic Regression produced higher F1-score, indicating a better balance between precision and recall.